# 05 · Language report card
**Time:** 15 min · **Goal:** measure how well the model handles *your* language, with a human judge.

Run the cells from top to bottom. **Do not share your key and do not upload private data.** Free-tier content may be used by Google to improve its products.

Get a key: https://aistudio.google.com/apikey

In [ ]:
# Setup: install the Gemini SDK and load your free AI Studio key
!pip -q install google-genai
import os, getpass
try:
    from google.colab import userdata          # Colab: left menu > Secrets > add GOOGLE_API_KEY
    API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    API_KEY = os.environ.get("GOOGLE_API_KEY") or getpass.getpass("Paste your AI Studio key: ")
from google import genai
from google.genai import types
client = genai.Client(api_key=API_KEY)
# Model names retire quickly. This tries several free-tier models and uses the first that works.
CANDIDATE_MODELS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-3.5-flash"]
def pick_model():
    for m in CANDIDATE_MODELS:
        try:
            client.models.generate_content(model=m, contents="ping")
            print("Using model:", m)
            return m
        except Exception as e:
            print("Skipping", m, "->", str(e)[:100])
    raise RuntimeError("No model worked. Check your key, account and the model list at https://ai.google.dev/gemini-api/docs/models")
MODEL = pick_model()
!pip -q install pandas
import pandas as pd

## Step 1 · Choose a language and run the test questions
Best practice: replace the English questions with questions written in *your* language (ask a friend or write them yourself). Testing only English prompts hides real problems.

**Scoring rubric (1 to 5):** 1 = unusable or wrong · 3 = understandable but with clear errors · 5 = correct and natural, as a native speaker would say it.

In [ ]:
LANGUAGE = "Swahili"   # change to your language, e.g. Luganda
QUESTIONS = [
    "My maize leaves are turning yellow. What should I do?",
    "How do I know if fall armyworm is attacking my maize?",
    "When should I plant beans after the rains start?",
    "How can I store maize so it does not rot?",
    "Which words in your answer are farming terms? List them.",
]
rows = []
for q in QUESTIONS:
    r = client.models.generate_content(model=MODEL, contents=f"Answer in {LANGUAGE}, briefly, as an advisor to a smallholder farmer: {q}")
    rows.append({"question": q, "answer": r.text})
df = pd.DataFrame(rows)
for c in ["understands", "farming_words", "correct", "natural", "dialect"]:
    df[c] = None
df

## Step 2 · A speaker of the language scores each answer 1 to 5
Columns: `understands`, `farming_words`, `correct`, `natural`, `dialect`. Run the cell and type the scores.

In [ ]:
# A native speaker reads each answer and types a score from 1 to 5
cols = ["understands", "farming_words", "correct", "natural", "dialect"]
for i in df.index:
    print(f"\nQ{i+1}: {df.loc[i,'question']}\nANSWER: {df.loc[i,'answer']}\n")
    for c in cols:
        df.loc[i, c] = int(input(f"  {c} (1-5): "))

## Step 3 · Summarise and export

In [ ]:
scores = df[["understands", "farming_words", "correct", "natural", "dialect"]].astype(float)
print(scores.mean().round(2))
df.insert(0, "language", LANGUAGE)
df.to_csv("report_card.csv", index=False)
print("Saved report_card.csv. Share your averages with the room.")

## Reflection
- Where did the model fail? Wrong words, wrong advice, or unnatural speech?
- Do not trust an AI judge for low-resource languages. Use human speakers.
- What data would fix it? See WAXAL, Common Voice and Masakhane in the slides.